# RedLamp: retraining and comparing results with the paper (KDD'25)

This notebook lets you:
1. Prepare all 5 datasets used in the paper: **UCR** (`anomaly_archive`), **AIOps** (`iops`), **SMD**, **SMAP**, **MSL**.
2. Choose a dataset and retrain it with `main.py` (the authors' original model and loss are kept unchanged).
3. Compute the paper's metrics: VUS-ROC, VUS-PR, Range F-score (RF), Range-AUC-ROC/PR, plus Accuracy for UCR.
4. Put the results next to RedLamp's numbers in **Table 3** and **Table 5** of the paper.

**How to use:**
- Run cells 1–2 once, then run cell 3 (configuration) and cell 4 (function definitions).
- **Option A:** run cell 5. The notebook processes each dataset in `DATASETS` in turn: it finishes training, evaluates and compares with the paper right away, saves the CSVs, and only then moves on to the next dataset.
- **Option B:** set `DATASET` in cell 3, then run cell 6 (train), cell 7 (evaluate) and cell 8 (summary).

It runs both locally (env `redlamp`, with the notebook inside the `RedLamp/` folder) and on Google Colab (Runtime → GPU).

### When Colab disconnects (or you want to use a new version of the code)
1. If you have a new version of the code, upload `RedLamp_colab.zip` (or `RedLamp_colab_full.zip`) to Drive, overwriting the old file. Otherwise skip this step.
2. Reconnect the runtime (GPU), then run **cell 1** (re-extracts the code if the zip changed and points `result/` to `MyDrive/RedLamp_result`), **cell 3** (configuration, keep the same `RUN_NAME`), **cell 4** (functions), then run **cell 5** again (or cell 6 if you only run one dataset).
3. The notebook resumes from where it stopped:
   - Finished jobs (with `anomaly_score.npy`, and for SMAP/MSL all channels) are skipped.
   - A job interrupted during training continues from the last epoch saved in `last.pt`.
   - A job that finished training but was interrupted during testing is only tested again, not retrained.
   - Finished datasets are re-evaluated with the current evaluation protocol.
4. Do not delete or rename the folder `MyDrive/RedLamp_result/<RUN_NAME>`. To retrain from scratch, change `RUN_NAME`.

## 1. Environment

In [ ]:
import os, sys, subprocess, collections

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    # Upload one of these two files to MyDrive:
    #   RedLamp_colab_full.zip (~165 MB, includes all 5 datasets) or RedLamp_colab.zip (~2 MB, data is downloaded in cell 2)
    from google.colab import drive
    drive.mount('/content/drive')
    # Look for the zip in any subfolder of MyDrive (and in shortcuts under .shortcut-targets-by-id, if any)
    import glob
    ZIP = None
    for name in ['RedLamp_colab_full.zip', 'RedLamp_colab.zip']:
        hits = sorted(glob.glob(f'/content/drive/MyDrive/**/{name}', recursive=True)
                      + glob.glob(f'/content/drive/.shortcut-targets-by-id/**/{name}', recursive=True))
        if hits:
            ZIP = hits[0]
            break
    if ZIP is None:
        raise FileNotFoundError('RedLamp_colab_full.zip or RedLamp_colab.zip not found in Google Drive. '
                                'Upload one of the two files to MyDrive (any folder), '
                                'or set the path directly: ZIP = "/content/drive/MyDrive/<folder>/RedLamp_colab_full.zip".')
    print('Using zip:', ZIP)
    PROJECT_DIR = '/content/RedLamp'
    # Re-extract when the zip on Drive changes (a new code version was uploaded). Only the code in /content is replaced;
    # results live on Drive (MyDrive/RedLamp_result) and the result/ symlink is removed first, so they are not touched.
    import shutil
    zip_id = f'{os.path.getsize(ZIP)}-{int(os.path.getmtime(ZIP))}'
    id_file = os.path.join(PROJECT_DIR, '.zip_id')
    if os.path.isdir(PROJECT_DIR) and (not os.path.isfile(id_file) or open(id_file).read() != zip_id):
        link = os.path.join(PROJECT_DIR, 'result')
        if os.path.islink(link):
            os.unlink(link)                 # only remove the symlink, never the contents on Drive
        if not os.path.isdir(link):          # a real result/ folder (not yet on Drive) is kept and handled below
            shutil.rmtree(PROJECT_DIR)
            print('New zip -> re-extracting the code')
    if not os.path.isdir(PROJECT_DIR):
        subprocess.run(['unzip', '-q', ZIP, '-d', '/content/'], check=True)
        open(id_file, 'w').write(zip_id)

    # result/ -> Google Drive: model, loss history, checkpoint (last.pt) and test results are saved directly to Drive,
    # so nothing is lost when Colab disconnects; running cell 5 (or 6) again resumes from the unfinished epoch/entity.
    RESULT_DRIVE = '/content/drive/MyDrive/RedLamp_result'
    os.makedirs(RESULT_DRIVE, exist_ok=True)
    link = os.path.join(PROJECT_DIR, 'result')
    if os.path.isdir(link) and not os.path.islink(link):
        for name in os.listdir(link):   # move old results (if any) to Drive
            if not os.path.exists(os.path.join(RESULT_DRIVE, name)):
                shutil.move(os.path.join(link, name), RESULT_DRIVE)
        shutil.rmtree(link)
    if not os.path.islink(link):
        os.symlink(RESULT_DRIVE, link)
    print('Results saved to:', RESULT_DRIVE)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'patool'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'TSB-AD'], check=True)
else:
    PROJECT_DIR = os.path.abspath('.')   # the notebook sits in the RedLamp/ folder

os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

import torch
print('Project:', PROJECT_DIR)
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2. Data preparation
This cell checks the data and downloads only the datasets that are missing. Datasets that already exist are skipped: locally, or when using `RedLamp_colab_full.zip`, this cell takes a few seconds. With the small zip on Colab, the cell downloads about 450 MB:
- SMAP and MSL come from `dataset/NASA.zip`.
- SMD is downloaded from the OmniAnomaly GitHub repo.
- UCR is downloaded from the UCR website.
- AIOps is the IOPS folder in TSB-UAD-Public.

In [ ]:
import prepare_data
prepare_data.prepare()   # or prepare_data.prepare(['smap']) to prepare a single dataset

## 3. Experiment configuration
| Dataset name | Dataset in the paper | What `ENTITIES='default'` runs |
|---|---|---|
| `anomaly_archive` | UCR (250 univariate series) | all **250 series** as in the paper. You can run a subset, e.g. `'001,002,...'`, but the result is marked PARTIAL and cannot be compared directly with the paper |
| `iops` | AIOps (29 KPIs) | all 29 KPIs |
| `smd` | SMD (28 machines, 38 features) | all 28 machines; or `'machine-1-1,machine-1-2'` |
| `smap` | SMAP (25 features) | one shared model for all 50 channels |
| `msl` | MSL (55 features) | one shared model for all 23 channels |

**There are two ways to run:**
- **Option A, run the datasets one after another (cell 5):**
  - The notebook goes through the datasets in `DATASETS` in order.
  - For each dataset: train all entities × seeds → evaluate right away → print the comparison table with the paper → save `summary_<dataset>.csv` and update `summary_all.csv`.
  - It moves to the next dataset only after the current one is finished. If a dataset has failed jobs, the notebook reports the error and continues instead of stopping everything.
- **Option B, run one dataset (cells 6–8):** uses the `DATASET` variable and runs each step separately: train, evaluate, summarize.

**`ENTITIES`:**
- `'default'` is exactly as in the paper.
- A string such as `'machine-1-1,machine-1-2'` applies only to `DATASET` (option B).
- A dict such as `{'smd': 'machine-1-1,machine-1-2'}` applies per dataset (both options).

The defaults follow **the paper's protocol**:
- 5 runs (`SEEDS = [0,1,2,3,4]`), reporting mean ± std over seeds.
- At most 100 epochs, early stopping with patience 10, 10% of train used for validation.
- AdamW lr 1e-3, batch 128, window 100.
- Window step 10 for AIOps/SMD/SMAP/MSL; for UCR it is {1, 10, 100} so that there are fewer than 10000 training windows.
- K = 12 augmentations, α = 0.1, β = 0.01, γ = 0.1, δ (FAA) = 0.05, as in Table 4.

`FIX_DOUBLE_SOFTMAX = True` is a **side experiment**, not used to reproduce the paper's numbers. The authors' official code applies `CrossEntropyLoss` to an output that has already gone through softmax (i.e. softmax twice), which differs from Eq.3. The paper's numbers were produced with the official code, so the default stays `False`. When set to `True`, the notebook adds the suffix `_fixsoftmax` to `RUN_NAME`.

**Saving and resuming:**
- On Colab, the `result/` folder points to `MyDrive/RedLamp_result`. Locally it is still `./result`.
- For each entity and each seed, the code saves:
  - `bestmodel.pkl`: the best weights according to validation.
  - The per-epoch loss history (`train_loss.txt`, `valid_loss.txt`, …).
  - `last.pt`: a checkpoint written after every epoch, with the model, optimizer, GradScaler, early-stopping state and RNG state.
  - `test_all/anomaly_score.npy`.
- If Colab disconnects, reopen it and run cells 1, 3, 4 and cell 5 (or cell 6):
  - Entities that already have `anomaly_score.npy` are skipped.
  - An entity interrupted during training continues from the last saved epoch with the same random state. On CPU the result is identical to an uninterrupted run.
- `last.pt` is deleted automatically after testing finishes.
- To retrain from scratch, change `RUN_NAME`.

**Running in parallel (`N_PARALLEL`):**
- The batch size stays 128. A large GPU (e.g. A100) uses only about 1 GB per process, so several jobs can run at the same time.
- Each job is one `main.py` process for exactly **one entity and one seed**. SMAP and MSL have only one entity each, so they are parallelised over seeds only (at most 5 jobs).
- Each entity is always reseeded with `--seed` when it starts, so an entity's result does not depend on whether it runs together with others or alone.
- On CPU, bit-identical results require the same number of CPU threads.
- Each job's log is in `result/<RUN_NAME>/logs/`. Skipping finished entities and resuming still work as before.

In [ ]:
DATASETS = ['msl', 'smap', 'smd', 'iops', 'anomaly_archive']   # option A: run one after another in this order
DATASET  = 'smap'        # option B: 'anomaly_archive' | 'iops' | 'smd' | 'smap' | 'msl'
ENTITIES = 'default'     # 'default' (as in the paper) | string 'a,b' for DATASET | dict {'smd': 'machine-1-1,machine-1-2'}
SEEDS    = [0, 1, 2, 3, 4]   # paper: mean of 5 runs
EPOCHS   = 100           # paper: at most 100 (early stopping, patience 10)
FIX_DOUBLE_SOFTMAX = False   # False = official code (used to reproduce the paper); True = CE as in Eq.3 (side experiment)
RUN_NAME = 'paper_repro' + ('_fixsoftmax' if FIX_DOUBLE_SOFTMAX else '')
GPU      = 0
VERBOSE  = False         # True: print the full main.py log (only when N_PARALLEL = 1)
N_PARALLEL = 1           # number of main.py processes run in parallel (each job = 1 entity × 1 seed); cell 4 prints a suggestion
EVAL_PROTOCOL = 'paper'  # 'paper' (default, matches the paper) | 'legacy' (old protocol: end-of-window alignment, oracle RF)

## 4. Shared functions (train / evaluate / summarize)
This cell only defines functions and runs nothing. It prints a suggested `N_PARALLEL` based on the number of CPUs and the free RAM. Peak RAM per process is about: SMAP 6 GB, MSL 5 GB, SMD 3 GB, AIOps 3 GB, UCR 2 GB.

In [ ]:
import time, glob, concurrent.futures
import pandas as pd
import evaluate as ev
pd.set_option('display.width', 200)

KEEP = ('Epoch:', 'best validation', 'Train', 'Test:', 'Finish', 'Device', 'exists', 'Resuming', 'Error', 'error', 'Traceback')
PER_PROC_GB = {'smap': 6, 'msl': 5, 'smd': 3, 'iops': 3, 'anomaly_archive': 2}   # measured peak RAM per process
OUT_DIR = os.path.join('result', RUN_NAME)

def entities_arg(ds):
    """ENTITIES value that applies to dataset ds: 'default' or a string 'a,b'."""
    if isinstance(ENTITIES, dict):
        return ENTITIES.get(ds, 'default')
    return ENTITIES if ds == DATASET else 'default'

def entity_list(ds):
    if ds in ('smap', 'msl'):
        return [None]                       # a single entity: parallel over seeds
    if entities_arg(ds) != 'default':
        return entities_arg(ds).split(',')
    if ds == 'anomaly_archive':
        return [str(i).zfill(3) for i in range(1, 251)]
    if ds == 'smd':
        from loaders.load import MACHINES
        return list(MACHINES)
    if ds == 'iops':
        return sorted({f.split('.')[0] for f in os.listdir('dataset/IOPS')})

def base_cmd(ds, seed):
    cmd = [sys.executable, '-u', 'main.py', '--dataset', ds, '--epoch', str(EPOCHS),
           '--seed', str(seed), '--run_name', RUN_NAME, '--gpu', str(GPU)]
    return cmd + (['--fix_double_softmax'] if FIX_DOUBLE_SOFTMAX else [])

N_CHANNELS = {'smap': 49, 'msl': 23}   # test_each folders written per seed (SMAP lists P-2 twice -> 49)

def job_done(ds, entity, seed):
    """Done = test_all/anomaly_score.npy exists (written last, atomically); for SMAP/MSL also every channel's test."""
    dirs = [d for d in glob.glob(f'{OUT_DIR}/{ds}/{entity or ds}/*/{seed}') if os.path.isfile(f'{d}/test_all/anomaly_score.npy')]
    if not dirs:
        return False
    if ds in N_CHANNELS:
        return len(glob.glob(f'{dirs[0]}/test_each/*/test_all/anomaly_score.npy')) >= N_CHANNELS[ds]
    return True

def run_main(ds, seed):
    """Sequential: one process for all entities of one seed. Returns True on success."""
    cmd = base_cmd(ds, seed)
    if entities_arg(ds) != 'default':
        cmd += ['--entities', entities_arg(ds)]
    elif ds == 'anomaly_archive':
        cmd += ['--entities', 'all']        # paper: mean over all 250 UCR series
    print('$', ' '.join(cmd))
    tail, t0 = collections.deque(maxlen=60), time.time()
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        tail.append(line)
        if VERBOSE or line.startswith(KEEP) or any(k in line for k in ('Error', 'Traceback')):
            print(line, end='')
    p.wait()
    if p.returncode != 0:
        print(''.join(tail))
        print(f'!!! main.py failed (exit {p.returncode}) on {ds} seed {seed}')
        return False
    print(f'--- {ds} seed {seed} done after {(time.time() - t0) / 60:.1f} min')
    return True

def run_job(ds, entity, seed, threads):
    """Parallel: one process for one entity × one seed, with its own log."""
    cmd = base_cmd(ds, seed) + (['--entities', entity] if entity is not None else [])
    log_dir = f'{OUT_DIR}/logs'
    os.makedirs(log_dir, exist_ok=True)
    log = f'{log_dir}/{ds}_{entity or ds}_seed{seed}.log'
    env = dict(os.environ, OMP_NUM_THREADS=str(threads), MKL_NUM_THREADS=str(threads))
    t0 = time.time()
    with open(log, 'w') as f:
        rc = subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT, env=env).returncode
    return entity, seed, rc, time.time() - t0, log

def suggest_parallel(ds):
    cpus = os.cpu_count() or 1
    try:
        import psutil
        free_gb = psutil.virtual_memory().available / 2**30
    except ImportError:
        free_gb = float('nan')
    by_ram = int(free_gb * 0.8 // PER_PROC_GB[ds]) if free_gb == free_gb else cpus
    return max(1, min(cpus, by_ram, len(entity_list(ds)) * len(SEEDS))), cpus, free_gb

def train_dataset(ds):
    """Train every entity × seed of ds (skips finished jobs, resumes from last.pt). Returns the list of failed jobs."""
    n_suggest, cpus, free_gb = suggest_parallel(ds)
    print(f'[{ds}] CPU: {cpus} | free RAM: {free_gb:.0f} GB | ~{PER_PROC_GB[ds]} GB/process -> suggested N_PARALLEL = {n_suggest}')
    if N_PARALLEL <= 1:
        return [f'{ds} seed {s}' for s in SEEDS if not run_main(ds, s)]
    jobs = [(e, s) for s in SEEDS for e in entity_list(ds) if not job_done(ds, e, s)]
    threads = max(1, (os.cpu_count() or 1) // N_PARALLEL)
    print(f'[{ds}] {len(jobs)} unfinished jobs, running {N_PARALLEL} processes in parallel ({threads} CPU threads per process)')
    t_start, done, failed = time.time(), 0, []
    with concurrent.futures.ThreadPoolExecutor(max_workers=N_PARALLEL) as pool:
        futures = [pool.submit(run_job, ds, e, s, threads) for e, s in jobs]
        for fut in concurrent.futures.as_completed(futures):
            entity, seed, rc, dt, log = fut.result()
            done += 1
            if rc != 0:
                failed.append(log)
            print(f'[{ds} {done}/{len(jobs)}] {entity or ds} seed={seed} '
                  f'{"OK" if rc == 0 else "FAILED (see " + log + ")"} after {dt/60:.1f} min | total {(time.time()-t_start)/60:.1f} min')
    return failed

def evaluate_dataset(ds):
    """Evaluate ds with the paper's metrics; saves metrics_<ds>.csv (per entity/seed) and summary_<ds>.csv."""
    df = ev.evaluate_run(RUN_NAME, ds, verbose=False, protocol=EVAL_PROTOCOL)
    os.makedirs(OUT_DIR, exist_ok=True)
    df.to_csv(os.path.join(OUT_DIR, f'metrics_{ds}.csv'), index=False)
    summary = ev.summarize(df, protocol=EVAL_PROTOCOL)
    summary.to_csv(os.path.join(OUT_DIR, f'summary_{ds}.csv'), index=False)
    for st in summary['status'].unique() if len(summary) else []:
        if not st.startswith('complete') and not st.startswith('reference'):
            print(f'Note [{ds}]:', st, '- does not yet follow the paper protocol (all entities × 5 seeds).')
    return df, summary

def refresh_summary_all():
    """Merge all existing summary_<ds>.csv files into summary_all.csv and return the ours/std/paper table."""
    parts = [pd.read_csv(os.path.join(OUT_DIR, f'summary_{d}.csv')) for d in ev.DATASETS
             if os.path.isfile(os.path.join(OUT_DIR, f'summary_{d}.csv'))]
    if not parts:
        return pd.DataFrame()
    all_summary = pd.concat(parts, ignore_index=True)
    all_summary.to_csv(os.path.join(OUT_DIR, 'summary_all.csv'), index=False)
    table = all_summary.set_index(['dataset', 'setting', 'metric'])[['ours (mean)', 'std (seeds)', 'paper']].unstack('metric')
    return table.swaplevel(axis=1).sort_index(axis=1)

for d in DATASETS:
    print(f'{d:16s} suggested N_PARALLEL = {suggest_parallel(d)[0]}')

## 5. Option A: run the datasets one after another (train → evaluate → compare with the paper → save)
- The notebook runs the datasets in `DATASETS` in order. Each dataset is fully trained, then evaluated, and the *ours / paper / diff* table is printed right away. The notebook then saves `result/<RUN_NAME>/summary_<dataset>.csv` and `metrics_<dataset>.csv` (details per entity/seed), and updates `summary_all.csv`.
- If a dataset has failed jobs, the notebook reports the error, still evaluates what has finished (marked PARTIAL), and moves on to the next dataset.
- If it is interrupted, run this cell again: finished parts are skipped.

In [ ]:
t_all = time.time()
problems = {}
for ds in DATASETS:
    t0 = time.time()
    print('=' * 100, f'\n>>> {ds}: train ({len(entity_list(ds))} entity × {len(SEEDS)} seed)')
    failed = train_dataset(ds)
    if failed:
        problems[ds] = failed
        print(f'!!! {ds}: {len(failed)} failed jobs -> {failed[:5]}')
    print(f'>>> {ds}: evaluate')
    try:
        _, summary = evaluate_dataset(ds)
        display(summary)
        refresh_summary_all()
    except Exception as e:
        problems.setdefault(ds, []).append(f'evaluation failed: {e!r}')
        print(f'!!! {ds}: evaluation failed: {e!r}')
    print(f'>>> {ds}: done after {(time.time() - t0) / 60:.1f} min (total {(time.time() - t_all) / 60:.1f} min)')

print('=' * 100)
print('Errors:', problems if problems else 'none')
display(refresh_summary_all().round(3))

## 6. Option B: train one dataset (`DATASET`)

In [ ]:
failed = train_dataset(DATASET)
if failed:
    raise RuntimeError(f'{len(failed)} failed jobs: {failed[:5]}')

## 7. Option B: evaluate `DATASET` with the paper's metrics and compare with Table 3/5
The default is `EVAL_PROTOCOL = 'paper'`. This protocol gives MSL results that match the paper with an average gap of about 0.01 (see cell 9):
- **SMAP/MSL:** the main row is **per-channel**, i.e. metrics are computed on each channel in `test_each` and then averaged. The `aggregate (reference)` row concatenates all channels; it is for reference only and is not compared with the paper.
- **Other datasets:** metrics are computed per entity, averaged over entities, then the mean and std over seeds are reported.
- **Score alignment:** each window's score is assigned to the **centre timestep of the window**, i.e. half a window of padding on each side. This is the TSB-UAD/VUS convention.
- **RF:** Range-F1 at the threshold **mean + 3·std**, as in the `metric_new` function of the original VUS/TSB-UAD package. The `RF (oracle)` column is RF at the best threshold, for reference only.
- Set `EVAL_PROTOCOL = 'legacy'` to go back to the old protocol (end-of-window alignment, oracle RF). Only re-evaluation is needed, no retraining.

In [ ]:
df, summary = evaluate_dataset(DATASET)
display(summary)

In [ ]:
# Detailed results per entity / seed
df.sort_values(['setting', 'entity', 'seed']).reset_index(drop=True)

## 8. Summary of all datasets evaluated with this `RUN_NAME` (`summary_all.csv`)

In [ ]:
# Re-evaluate every dataset that has results, then merge the tables
for d in ev.DATASETS:
    if os.path.isdir(os.path.join(OUT_DIR, d)):
        evaluate_dataset(d)
display(refresh_summary_all().round(3))

## 9. Diagnosing the gap with the paper (MSL/SMAP)
This cell retrains nothing; it only re-scores the results already in `result/<RUN_NAME>/` in several ways and compares them with the paper:
- **How results are combined:**
  - aggregate (`test_all`);
  - mean over channels (`test_each`);
  - concatenating the channels' scores.
- **Metric implementation:**
  - TSB-AD, as in `evaluate.py`;
  - the original VUS package (pip `vus`, installed with `--no-deps`) with slidingWindow = find_length / 100 / 50.
- **RF:** oracle threshold or mean + 3·std threshold.
- **Score alignment:** each window's score assigned to the end / centre / start timestep of the window.
- Add `--baseline` to also score IsolationForest, for comparison with the IsolationForest row in the paper.

Results are saved to `result/<RUN_NAME>/diag/<dataset>_gap.csv`. On CPU, each MSL seed takes about 7 minutes.

In [ ]:
!{sys.executable} diag_msl_gap.py --run_name {RUN_NAME} --dataset msl
# !{sys.executable} diag_msl_gap.py --run_name {RUN_NAME} --dataset smap --alignments end centre

## Notes on the comparison
- **Metrics** use the TSB-AD implementation (`TSB_AD.evaluation.basic_metrics`):
  - Range-AUC: parameter `window = 100`. TSB-AD extends each anomalous segment by `window // 2` on each side, so the buffer is 50, i.e. exactly half the window as stated in the paper.
  - VUS: `slidingWindow = 100`. This is TSB-AD's default and also equals the model's window L. VUS takes the volume under the surface as the buffer goes from 0 to 50.
  - RF: Range-F1 (Tatbul 2018, alpha = 0.2) at the threshold mean + 3·std of the score, like the `metric_new` function in the original VUS/TSB-UAD package. For MSL, RF = 0.244 vs 0.254 in the paper.
  - `RF (oracle)`: Range-F1 at the best threshold. For reference only, since it gives much higher scores than the paper.
  - Acc (UCR) is 1 if the point with the highest score falls inside the anomalous segment.
- **Score alignment**: each window has length 100 and step 1, and its score is assigned to the **centre timestep of the window** (protocol `paper`). For MSL, switching from end to centre alignment raises VUS-PR from 0.372 to 0.469.
- **MSL** (5 seeds): aggregate only reaches a VUS-PR of about 0.14, while per-channel with centre alignment reaches 0.469, vs 0.482 in the paper. Details in `diag_msl_gap.py` (cell 9).
- **The numbers will not match exactly**, because the augmentation is random, the GPU/torch versions differ, and the paper averages 5 runs. A gap of about ±0.02–0.05 is normal.
- **Time**:
  - SMAP takes about 3 minutes/epoch on a laptop CPU; much faster on GPU.
  - UCR with all 250 series × 5 seeds takes very long (needs a GPU). It can be run in parts and resumed, since finished entities are skipped. Results can be compared with the paper only when the `status` column says `complete`.
- **RAM**: the test step now slices windows on the fly instead of building them all up front. Peak RAM for SMAP is only about 6 GB (previously about 23 GB), MSL about 5 GB, one SMD machine about 3 GB, so free Colab (12.7 GB) can run it. Results are identical to the old code.
- **Output files** in `test_all/`:
  - `anomaly_score.npy`: the file used for evaluation.
  - `mse.npy`: reconstruction error of each window.
  - `pred_label.npy`, `label.npy`, `enc.npy`.
  - `input_last.npy`: the value at the last timestep of each window.
  - The full `input.npy`, `pred.npy`, `anomaly_mask.npy` are no longer saved (for SMAP each file is about 3.9 GB).